# Quadrotor 2D vision Koopman training and evaluation

This notebook runs a reference end-to-end experiment for vision-based Koopman learning on `quadrotor_2d`.

It uses the same project code as the command-line entry points. The notebook is intended for inspection, diagnostics, and figure generation, not as a separate implementation of the training pipeline.

Expected runtime for full training is higher than the sensor case and depends strongly on hardware and image configuration.

## Workflow

1. Select the system, modality, dataset, and run identifier.
2. Optionally launch training.
3. Load the trained Koopman model.
4. Inspect the learned latent dynamics.
5. Run open-loop evaluation.
6. Optionally inspect closed-loop behavior.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import torch

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "src" / "KoNAMIC").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

sys.path.insert(0, str(PROJECT_ROOT / "src"))

from KoNAMIC import config, paths, utils
from KoNAMIC.core.systems import create_system
from KoNAMIC.koopman.models import load_koop_model_for_eval
from KoNAMIC.koopman.models.model_config import ModelConfig
from KoNAMIC.pipelines.data_preparation import VisionPreparationConfig

%matplotlib inline

In [ ]:
SYSTEM_NAME = "quadrotor_2d"
MODALITY = config.Modality.VISION
LATENT_DYNAMICS = "linear"
RUN_SENSOR_DATASET_GENERATION = False
RUN_VISION_DATASET_GENERATION = False
SENSOR_DATASET_CONTROLLER = "pid"
VISION_SYSTEM_DIM = 2
DATASET_STAMP = "TODO_DATASET_STAMP"
RUN_ID = "notebook_quad2d_vision_demo"
RUN_STATUS = "interim"
STAMP_RUN = None  # Example: "lin_notebook_quad2d_vision_demo_2026-07-05_14-30-00"
TRAINING_CONTROLLER_VARIANT = None
SEED = 0
EPOCH = None  # Set to an integer checkpoint epoch when needed.
OPEN_LOOP_PHASE = "val_2"
OPEN_LOOP_NUM_STEPS = 100
OPEN_LOOP_NUM_ROLLOUTS = 1

RUN_OPEN_LOOP_EVAL = False
RUN_CLOSED_LOOP_EVAL = False
CLOSED_LOOP_CONTROLLER = "kmpc"
CLOSED_LOOP_CONTROLLER_VARIANT = None
CLOSED_LOOP_SCENARIO_LEVEL = "smooth"
RUN_TRAINING = False

In [ ]:
utils.set_seed(SEED)

system_spec = create_system(SYSTEM_NAME)

print(f"project_root: {PROJECT_ROOT}")
print(f"system_name: {system_spec.system_name}")
print(f"x_dim: {system_spec.x_dim}")
print(f"u_dim: {system_spec.u_dim}")
print(f"modality: {MODALITY.key}")

## Optional sensor dataset generation

Set `RUN_SENSOR_DATASET_GENERATION = True` to generate a fresh sensor dataset for `quadrotor_2d`.

The vision dataset renderer uses these sensor trajectories as input. After generation, this cell updates `DATASET_STAMP` with the timestamped dataset directory that was created.

In [ ]:
if RUN_SENSOR_DATASET_GENERATION:
    import subprocess

    datasets_root = paths.build_dataset_paths(SYSTEM_NAME, "placeholder").root.parent
    before = {p.name for p in datasets_root.iterdir() if p.is_dir()} if datasets_root.exists() else set()

    sensor_dataset_cmd = [
        sys.executable,
        str(PROJECT_ROOT / "entrypoints" / "generate_sensor_dataset.py"),
        "--system-name", SYSTEM_NAME,
        "--controller", SENSOR_DATASET_CONTROLLER,
        "--seed", str(SEED),
    ]

    print("Running:")
    print(" ".join(sensor_dataset_cmd))
    subprocess.run(sensor_dataset_cmd, cwd=PROJECT_ROOT, check=True)

    after = {p.name for p in datasets_root.iterdir() if p.is_dir()}
    new_stamps = sorted(after - before)
    if not new_stamps:
        raise FileNotFoundError(f"No new dataset directory detected under {datasets_root}")

    DATASET_STAMP = new_stamps[-1]
    print(f"Detected DATASET_STAMP: {DATASET_STAMP}")
else:
    print("Sensor dataset generation skipped. Set RUN_SENSOR_DATASET_GENERATION = True to generate sensor trajectories.")

## Optional vision dataset generation

Set `RUN_VISION_DATASET_GENERATION = True` to render a fresh vision dataset for `quadrotor_2d` from an existing sensor dataset.

The notebook delegates rendering to `entrypoints/generate_vision_dataset.py`. `DATASET_STAMP` must identify the dataset directory that contains the sensor trajectories.

In [ ]:
if RUN_VISION_DATASET_GENERATION:
    if DATASET_STAMP == "TODO_DATASET_STAMP":
        raise RuntimeError("Set DATASET_STAMP to an existing sensor dataset before generating vision data.")

    import subprocess

    dataset_cmd = [
        sys.executable,
        str(PROJECT_ROOT / "entrypoints" / "generate_vision_dataset.py"),
        "--system-name", SYSTEM_NAME,
        "--dataset-stamp", DATASET_STAMP,
        "--system-dim", str(VISION_SYSTEM_DIM),
    ]

    print("Running:")
    print(" ".join(dataset_cmd))
    subprocess.run(dataset_cmd, cwd=PROJECT_ROOT, check=True)
else:
    print("Vision dataset generation skipped. Set RUN_VISION_DATASET_GENERATION = True to render a dataset.")

## Optional training

Set `RUN_TRAINING = False` to launch the standard training entry point from this notebook.

The notebook does not reimplement the training loop. It delegates training to `entrypoints/train_model.py`, then later cells can load and inspect the resulting run.

In [ ]:
if RUN_TRAINING:
    import subprocess

    if STAMP_RUN is None:
        prefix_by_dynamics = {"linear": "lin", "bilinear": "bilin"}
        STAMP_RUN = f"{prefix_by_dynamics[LATENT_DYNAMICS]}_{RUN_ID}_{paths.make_timestamp()}"

    train_cmd = [
        sys.executable,
        str(PROJECT_ROOT / "entrypoints" / "train_model.py"),
        "--modality", MODALITY.key,
        "--system-name", SYSTEM_NAME,
        "--latent-dynamics", LATENT_DYNAMICS,
        "--dataset-stamp", DATASET_STAMP,
        "--id", RUN_ID,
        "--stamp-run", STAMP_RUN,
        "--seed", str(SEED),
    ]
    if TRAINING_CONTROLLER_VARIANT is not None:
        train_cmd.extend(["--controller-variant", TRAINING_CONTROLLER_VARIANT])

    print("Running:")
    print(" ".join(train_cmd))
    subprocess.run(train_cmd, cwd=PROJECT_ROOT, check=True)

    run_paths = paths.build_run_paths(
        modality=MODALITY.key,
        system_name=SYSTEM_NAME,
        run_status=RUN_STATUS,
        stamp_run=STAMP_RUN,
    )
    print(f"STAMP_RUN: {STAMP_RUN}")
    print(f"Run directory: {run_paths.run_dir}")
else:
    print("Training skipped. Set RUN_TRAINING = True to launch training.")

## Select a trained run

Set `STAMP_RUN` to the timestamped run directory to inspect.

If `RUN_TRAINING = False`, use the run directory created by the training command. Otherwise, select an existing run from `outputs/`.

In [ ]:
runs_base_dir = paths.build_base_output_dir(
    modality=MODALITY.key,
    run_status=RUN_STATUS,
    system_name=SYSTEM_NAME,
)
runs_dir = runs_base_dir / "runs"

print(f"runs_dir: {runs_dir}")

if STAMP_RUN is None:
    print("Set STAMP_RUN to one of the available run directories before loading a model.")
    if runs_dir.exists():
        run_dirs = sorted(
            [p for p in runs_dir.iterdir() if p.is_dir()],
            key=lambda p: p.stat().st_mtime,
            reverse=True,
        )
        print("\nMost recent runs:")
        for run_path in run_dirs[:10]:
            print(f"- {run_path.name}")
    else:
        print("No runs directory found yet.")
    run_paths = None
else:
    run_paths = paths.build_run_paths(
        modality=MODALITY.key,
        system_name=SYSTEM_NAME,
        run_status=RUN_STATUS,
        stamp_run=STAMP_RUN,
    )
    if not run_paths.run_dir.exists():
        raise FileNotFoundError(f"Run directory does not exist: {run_paths.run_dir}")
    print(f"Selected run directory: {run_paths.run_dir}")

## Load the trained model

This cell loads the saved run configuration, resolves model dimensions from the selected system, selects a checkpoint epoch, and loads the trained Koopman model with its dataset scalers.

In [ ]:
if run_paths is None:
    raise RuntimeError("Set STAMP_RUN and rerun the run-selection cell before loading a model.")

run_config = config.load_yaml(run_paths.run_dir / "config.yaml")
model_raw = run_config["model"] if "model" in run_config else run_config
vision_preparation_raw = run_config["data_preparation"]

data_preparation_config = VisionPreparationConfig.from_dict(vision_preparation_raw)
model_config = ModelConfig.from_dict(model_raw)
model_config = model_config.with_system_dimensions(
    x_dim=system_spec.x_dim,
    u_dim=system_spec.u_dim,
).with_delay(data_preparation_config.postprocessing.delay)
model_config = model_config.with_consistent_latent_dynamics()

if EPOCH is None:
    checkpoint_paths = sorted(run_paths.checkpoints_dir.glob("model_epoch_*.pt"))
    if not checkpoint_paths:
        raise FileNotFoundError(f"No checkpoints found in {run_paths.checkpoints_dir}")
    epoch = max(int(path.stem.rsplit("_", 1)[-1]) for path in checkpoint_paths)
else:
    epoch = int(EPOCH)

koop_model, data_scalers = load_koop_model_for_eval(
    MODALITY,
    model_config,
    epoch,
    run_paths.run_dir,
)

print(f"Loaded epoch: {epoch}")
print(f"model type: {type(koop_model).__name__}")
print(f"x_dim: {model_config.z_dynamics.x_dim}")
print(f"u_dim: {model_config.z_dynamics.u_dim}")
print(f"z_dim: {model_config.z_dynamics.z_dim}")
print(f"latent dynamics: {model_config.z_dynamics.model}")
print(f"delay: {data_preparation_config.postprocessing.delay}")
print(f"num_views: {getattr(koop_model, 'num_views', 'unknown')}")

## Inspect Koopman matrices

Extract the learned latent dynamics matrices and inspect their dimensions, norms, and eigenvalues.

In [ ]:
with torch.no_grad():
    A_t, B_t = koop_model.construct_koop_matrices()

A = A_t.detach().cpu().numpy()
B = B_t.detach().cpu().numpy()
eigvals = np.linalg.eigvals(A)

print(f"A shape: {A.shape}")
print(f"B shape: {B.shape}")
print(f"||A||_F: {np.linalg.norm(A):.4f}")
print(f"||B||_F: {np.linalg.norm(B):.4f}")
print(f"max |eig(A)|: {np.max(np.abs(eigvals)):.4f}")

fig, ax = plt.subplots(figsize=(5, 5))
unit_circle = plt.Circle((0.0, 0.0), 1.0, color="black", fill=False, linestyle="--", linewidth=1)
ax.add_artist(unit_circle)
ax.scatter(eigvals.real, eigvals.imag, s=28)
ax.axhline(0.0, color="0.8", linewidth=1)
ax.axvline(0.0, color="0.8", linewidth=1)
ax.set_aspect("equal", adjustable="box")
ax.set_xlabel("Real")
ax.set_ylabel("Imaginary")
ax.set_title("Eigenvalues of A")
plt.show()

## Optional open-loop evaluation

The current open-loop entry point is sensor-only. Keep this section disabled unless vision support is added to `entrypoints/run_open_loop_simulation.py`.

For now, this vision notebook focuses on dataset rendering, training, model loading, and matrix inspection.

In [ ]:
if RUN_OPEN_LOOP_EVAL:
    raise NotImplementedError(
        "Vision open-loop evaluation is not exposed by entrypoints/run_open_loop_simulation.py yet."
    )
else:
    print("Vision open-loop evaluation skipped.")

## Optional closed-loop evaluation

Closed-loop evaluation for vision Koopman models is not the default path in this notebook. KMPC currently expects reference projection support that may be unavailable for vision models.

Keep this section disabled unless the closed-loop vision path has been validated.

In [ ]:
if RUN_CLOSED_LOOP_EVAL:
    raise NotImplementedError(
        "Closed-loop evaluation for vision Koopman models should be validated before launching it from this notebook."
    )
else:
    print("Vision closed-loop evaluation skipped.")